# Setup and Data Preparation

This section sets up the environment by cloning the necessary repository, installing dependencies, and extracting the challenge data. It also prints the GPU information and verifies the presence of key data files.

In [1]:
import os, zipfile, shutil

!git clone https://github.com/lennardpische/staix26_seasthemoment.git
%cd staix26_seasthemoment
!git checkout transformer-lenny
!pip install -r requirements.txt -q

import torch
print("GPU:", torch.cuda.get_device_name(0))

ZIP_PATH = "/content/staix-challenge.zip"
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall("/content/staix26_seasthemoment/")

for path in ["train/dose_sys_train.csv","train/covariates.csv",
              "val/covariates.csv","sample_submission.csv",
              "train/images/mat_density","val/images/mat_density"]:
    print(f"{'✓' if os.path.exists(path) else '✗'} {path}")

Cloning into 'staix26_seasthemoment'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 209 (delta 108), reused 160 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 396.27 KiB | 8.81 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/staix26_seasthemoment
Branch 'transformer-lenny' set up to track remote branch 'transformer-lenny' from 'origin'.
Switched to a new branch 'transformer-lenny'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.7 MB/s eta 0:00:00
GPU: NVIDIA A100-SXM4-80GB
✓ train/dose_sys_train.csv
✓ train/covariates.csv
✓ v

# Vectorization

This step runs the `vectorize.py` script, which processes the text and image data to generate embeddings. These embeddings are then saved as CSV files for later use in model training.

In [2]:
!python -m src.vectorize

Loading data...

Text embeddings (all-mpnet-base-v2)...
  Encoding train text...
modules.json: 100% 349/349 [00:00<00:00, 1.46MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 755kB/s]
README.md: 100% 11.6k/11.6k [00:00<00:00, 31.1MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 254kB/s]
config.json: 100% 571/571 [00:00<00:00, 3.78MB/s]
model.safetensors: 100% 438M/438M [00:01<00:00, 230MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 6008.08it/s]
tokenizer_config.json: 100% 363/363 [00:00<00:00, 1.85MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 22.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 73.7MB/s]
special_tokens_map.json: 100% 239/239 [00:00<00:00, 1.17MB/s]
config.json: 100% 190/190 [00:00<00:00, 1.24MB/s]
Batches: 100% 62/62 [00:09<00:00,  6.67it/s]
  Encoding val text...
Loading weights: 100% 199/199 [00:00<00:00, 4986.57it/s]
Batches: 100% 6/6 [00:00<00:00,  6.57it/s]
  Saved embeddings/text_train.csv  (3927, 770)
  Saved embeddings/text_

# Model Prediction

This section first pulls the latest changes from the repository to ensure the most up-to-date model code is used. Then, it executes the `predict.py` script to train the transformer model and generate predictions, saving them to `expert_transformer.csv`.

In [7]:
!git pull
!rm -rf checkpoints/   # force full retrain with new features
!python -m src.predict

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 629 bytes | 629.00 KiB/s, done.
From https://github.com/lennardpische/staix26_seasthemoment
   546c136..ab890eb  transformer-lenny -> origin/transformer-lenny
Updating 546c136..ab890eb
Fast-forward
 src/features.py | 7 +++----
 1 file changed, 3 insertions(+), 4 deletions(-)
Loading data...
  Train: (31416, 22)  Val: (357, 19)  Template: 918 rows
Building features...
  Numeric: 31  Text SVD: 32  Image PCA: 64
Training (GroupKFold × 5)...
  Device: cuda
  Checkpoints: /content/staix26_seasthemoment/checkpoints
  Fold 1/5  (0s) ... transformer
    → saved to checkpoints/fold_0.npz
  Fold 2/5  (120s) ... transformer
    → saved to checkpoints/fold_1.npz
  Fold 3/5  (194s) ... transformer
    → saved to checkpoints/fold_2.npz
  Fold 4/5  (274s) ... transformer
   

# Download Predictions

This cell facilitates downloading the generated prediction file (`expert_transformer.csv`) to your local machine.

In [9]:
from google.colab import files
files.download('expert_transformer.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Data Loading and Feature Pipeline

Here, the notebook loads the generated text and image embeddings, along with the raw training and validation data. It then applies a feature pipeline to transform the data into a format suitable for the model, checking the dimensions of the processed features.

In [5]:
import pandas as pd
import numpy as np

txt = pd.read_csv('embeddings/text_train.csv', nrows=2)
img = pd.read_csv('embeddings/img_train.csv', nrows=2)
print("Text embedding dims:", len(txt.columns) - 2)   # should be 768
print("Image embedding dims:", len(img.columns) - 2)
import sys
sys.path.insert(0, '/content/staix26_seasthemoment')
from src.data_loader import load_train, load_val
from src.features import FeaturePipeline

train = load_train()
val = load_val()
pipe = FeaturePipeline()
td, vd = pipe.fit_transform(train, val, data_root=None)

print("\nActual dims hitting the model:")
print("Numeric:", td['numeric'].shape[1])
print("Text:   ", td['text'].shape[1])    # 768 = cache loaded, 32 =
print("Image:  ", td['image'])

train = pd.read_csv('train/dose_sys_train.csv')
print(train.groupby('overdose_category')['rate_per_10000_ed_visits'].describe())

Text embedding dims: 768
Image embedding dims: 1408

Actual dims hitting the model:
Numeric: 36
Text:    768
Image:   [[-0.11114502  0.3190918   0.09619141 ...  0.17712402  0.56152344
  -0.18969727]
 [-0.09338379 -0.15344238  0.5683594  ...  0.3293457  -0.11877441
  -0.0723877 ]
 [-0.15405273  0.4104004   0.5878906  ...  0.02815247 -0.17907715
  -0.1607666 ]
 ...
 [ 0.19592285  0.09014893  0.22424316 ...  0.14318848 -0.15539551
  -0.19567871]
 [ 0.9091797  -0.18884277  1.0878906  ...  0.34838867  0.06622314
  -0.17541504]
 [ 0.00231361 -0.11730957 -0.03601074 ...  0.41455078  0.15576172
   0.10412598]]
                    count       mean       std   min     25%    50%     75%  \
overdose_category                                                             
all_drugs          3927.0  22.820397  9.905307  6.37  16.105  20.24  26.820   
all_opioids        3927.0  10.304416  4.473696  2.96   7.270   9.26  12.035   
all_stimulants     3927.0   5.219740  2.382680  1.20   3.630   4.67   6.20

# Constraint Validation

This final section loads the `expert_transformer.csv` predictions and `sample_submission.csv` to verify that the predictions adhere to specified constraints (e.g., 'all_drugs' being greater than 'all_opioids' and 'all_stimulants'). It also provides a descriptive summary of the predictions.

In [6]:
  import pandas as pd
  import numpy as np

  df = pd.read_csv('expert_transformer.csv')

  # Merge to check constraint violations
  template = pd.read_csv('sample_submission.csv')
  wide = df.merge(template[['row_id','overdose_category']], on='row_id')
  wide = wide.pivot_table(index=wide.index // 3,
  columns='overdose_category', values='rate_per_10000_ed_visits')

  violations = (wide['all_drugs'] < wide['all_opioids']).sum() + \
               (wide['all_drugs'] < wide['all_stimulants']).sum()
  print(f"Constraint violations: {violations}")  # should be 0
  print(wide.describe())

Constraint violations: 0
overdose_category   all_drugs  all_opioids  all_stimulants
count              306.000000   306.000000      306.000000
mean                23.026393    10.322720        5.978691
std                  8.561806     3.891236        2.066078
min                 12.850067     5.883350        3.300647
25%                 16.599479     7.473981        4.432030
50%                 20.379748     9.214393        5.420294
75%                 26.455219    11.730991        6.827621
max                 52.132153    26.262608       14.047491
